# 03 — Baselines

Same problems, same single VLM reading where one is used:
- **contrast** — CV verifier only (free; no language model).
- **direct** — gpt-4o looks at the classic montage once and states the rule (the vision model reasons itself).
- **router** — the CV rule when the CV verifier proves one, otherwise the direct answer.
- **single agent** — one gpt-oss call reads the full eye + ruler record and states the rule.
- **self-consistency** — the single agent sampled N times, majority vote; N is matched to the pipeline's mean
  number of calls, so it has the same compute as the multi-agent debate.

Single agent and self-consistency use the same "clear measured rule first" step as the pipeline, so the only
difference from the pipeline is how the language model reasons.

In [ ]:
%run ./00_common.ipynb

In [ ]:
PROBLEMS = PROBLEM_SETS[os.environ.get("PROBLEM_SET", "SMOKE")]
AGENT = "openai/gpt-oss-120b"
DIRECT_PROMPT = ('This is a Bongard problem. The TOP row shows 6 LEFT panels that all share one hidden visual rule; '
    'the BOTTOM row shows 6 RIGHT panels that break it (usually they show the opposite). Find the single simplest rule. '
    'Prefer short, concrete descriptions (shape type, size, orientation, position, number, relation). '
    'Reply json: {"left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>"}.')
SINGLE_PROMPT = """You are a visual pattern analyst. You may use every recorded property of the panels.
This is a Bongard problem: the 6 LEFT panels all follow ONE hidden rule; the 6 RIGHT panels break it (usually they show the opposite).
LEFT panels:
{left}
RIGHT panels:
{right}

Find the simplest rule that is true for ALL 6 left panels and false for ALL 6 right panels. Check it panel by panel.
You MUST give a concrete rule. Say each side as a short visual concept a person would use (3-10 words), e.g. '<property> figures' or '<object A> <relation> <object B>'. Name what IS on each side, not what is missing, unless absence itself is the rule. Do NOT mention numbers, thresholds, coordinates, 'ruler', 'blob', 'eye', 'record' or panel labels.
Return JSON only: {{"left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>", "confidence": 0.0}}"""

contrast = {}
for bp in PROBLEMS:                        # original contrast_only_baseline: verified rule, else best contrast guess
    pool = load_cv(bp)
    cr = compute_contrast(pool)
    r = propose_verified_rule(pool, contrast=cr)
    contrast[bp] = {"bp": bp, "rule": r["nl"] if r else best_contrast_nl(pool, cr), "verified": bool(r),
                    "calls": 0, "cost_usd": 0.0}
save_predictions("contrast", contrast)
print("contrast: CV proves a rule on", sum(v["verified"] for v in contrast.values()), "of", len(PROBLEMS))

In [ ]:
direct = load_predictions(f"direct__{VLM_MODEL}")
METER.tag = f"direct__{VLM_MODEL}"
for bp in PROBLEMS:
    if bp in direct:
        continue
    png, _ = std_montage(bp)
    before = METER.total
    out = vlm_json(DIRECT_PROMPT, png, max_tokens=300)
    rule = make_rule(str(out.get("left", "")), str(out.get("right", "")))
    direct[bp] = {"bp": bp, "rule": rule, "verified": bool(verify_text_rule(rule, load_cv(bp))),
                  "calls": 1, "cost_usd": METER.total - before}
    save_predictions(f"direct__{VLM_MODEL}", direct)
router = {bp: dict(contrast[bp] if contrast[bp]["verified"] else direct[bp], bp=bp) for bp in PROBLEMS}
save_predictions("router__cv-else-direct", router)
print("direct + router saved")

## Single agent and self-consistency (same records as the pipeline)

In [ ]:
spec = load_predictions("pipeline__specialist__gpt-oss-120b__ask_eye-judge")
N_SC = max(3, round(np.mean([r["calls"] for r in spec.values()]))) if spec else 12
print("self-consistency samples per problem (matched to the pipeline's mean calls):", N_SC)

def one_answer(bp, temperature):
    left, right = records_block(panel_records(bp, "hybrid"))
    try:
        out = text_json(SINGLE_PROMPT.format(left=left, right=right), model=AGENT, max_tokens=1500, temperature=temperature)
    except BudgetExceeded:
        raise
    except Exception:
        out = {}
    return as_answer(out | {"has_rule": True})

def majority(answers):
    ruled = [a for a in answers if a["has_rule"]]
    if not ruled:
        return None, 0
    groups = []
    for a in ruled:
        key = (a["left"] + " | " + a["right"]).lower()
        for g in groups:
            if SequenceMatcher(None, key, (g[0]["left"] + " | " + g[0]["right"]).lower()).ratio() > 0.6:
                g.append(a); break
        else:
            groups.append([a])
    best = max(groups, key=lambda g: (len(g), max(x["confidence"] for x in g)))
    return max(best, key=lambda x: x["confidence"]), len(best)

for arm, n_samples, temp in [(f"single__{AGENT.split('/')[-1]}", 1, 0.0), (f"self-consistency__{AGENT.split('/')[-1]}", N_SC, 0.7)]:
    rows = load_predictions(arm)
    METER.tag = arm
    for bp in PROBLEMS:
        if bp in rows:
            continue
        before, calls0 = METER.total, METER.calls
        cv_rule = propose_verified_rule(load_cv(bp), contrast=compute_contrast(load_cv(bp)))
        if cv_rule and cv_rule_margin(cv_rule, load_cv(bp)) < CV_MIN_MARGIN:
            cv_rule = None                                       # a narrow split is not trusted over the agent
        answers = [one_answer(bp, temp) for _ in range(n_samples)]
        pick, votes = majority(answers)
        agent_rule = pick["rule"] if pick else ""
        final = cv_rule["nl"] if cv_rule else agent_rule
        rows[bp] = {"bp": bp, "rule": final, "rule_agent": agent_rule,
                    "verified": bool(cv_rule) or bool(verify_text_rule(final, load_cv(bp))),
                    "votes": votes, "samples": [a["rule"] for a in answers], "calls": METER.calls - calls0,
                    "cost_usd": METER.total - before, "perception_cost_usd": perception_cost(bp)}
        save_predictions(arm, rows)
        print(f"  {arm:28s} {bp}  {rows[bp]['rule'][:90]}")
METER.report()